In [ ]:
# # TripSense — Recommendation Engine Evaluation

# ## Objective
# Evaluate three recommendation approaches for tourist destinations in Andhra Pradesh:

# 1. Content-Based Filtering
# 2. Collaborative Filtering
# 3. Hybrid Recommendation

# ## Evaluation Metric
# Precision@K measures the proportion of recommended destinations
# in the top K that are relevant to the user.

# A rating of 4 or 5 is considered a positive interaction.

# We evaluate Precision@5 and Precision@10 using held-out
# positive interactions.

In [1]:
import pandas as pd
import numpy as np

from content_recommender import ContentBasedRecommender
from collaborative_recommender import CollaborativeRecommender
from hybrid_recommender import HybridRecommender

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
places = pd.read_csv("/content/tripsense_places_v2.csv")
reviews = pd.read_csv("/content/tripsense_reviews_dataset.csv")

reviews["rating"] = pd.to_numeric(
    reviews["rating"],
    errors="coerce"
)

reviews = reviews.dropna(
    subset=["reviewer_name", "place_id", "rating"]
)

print("Places dataset:", places.shape)
print("Reviews dataset:", reviews.shape)
print("Unique users:", reviews["reviewer_name"].nunique())

Places dataset: (444, 57)
Reviews dataset: (2220, 11)
Unique users: 70


In [3]:
rng = np.random.RandomState(42)

train_indices = []
test_indices = []

for user_id, user_data in reviews.groupby("reviewer_name"):

    positive_indices = user_data.index[
        user_data["rating"] >= 4
    ].to_numpy()

    if len(positive_indices) >= 2:
        shuffled = rng.permutation(positive_indices)

        n_test = max(1, int(len(shuffled) * 0.20))
        n_test = min(n_test, len(shuffled) - 1)

        test_indices.extend(shuffled[:n_test])
        train_indices.extend(shuffled[n_test:])

        other_indices = user_data.index[
            ~user_data.index.isin(positive_indices)
        ].to_numpy()

        train_indices.extend(other_indices)

    else:
        train_indices.extend(user_data.index.to_numpy())

train_set = reviews.loc[sorted(train_indices)].copy()

test_set = reviews.loc[sorted(test_indices)].copy()
test_set = test_set[test_set["rating"] >= 4].copy()

print("Training interactions:", len(train_set))
print("Test interactions:", len(test_set))
print("Users in training:", train_set["reviewer_name"].nunique())
print("Users in test:", test_set["reviewer_name"].nunique())

Training interactions: 1977
Test interactions: 243
Users in training: 70
Users in test: 70


In [4]:
content_model = ContentBasedRecommender(places)

collaborative_model = CollaborativeRecommender(
    train_set
)

hybrid_model = HybridRecommender(
    places_df=places,
    content_recommender=content_model,
    collaborative_recommender=collaborative_model
)

print("Content-based model: Ready")
print("Collaborative model: Ready")
print("Hybrid model: Ready")

Content-based model: Ready
Collaborative model: Ready
Hybrid model: Ready


In [5]:
def build_user_interests(user_id, train_data, places_df):

    user_history = train_data[
        (train_data["reviewer_name"] == user_id)
        & (train_data["rating"] >= 4)
    ]

    if user_history.empty:
        return ""

    liked_place_ids = user_history["place_id"].tolist()

    liked_places = places_df[
        places_df["place_id"].isin(liked_place_ids)
    ]

    text_parts = []

    for _, row in liked_places.iterrows():
        for column in [
            "tags_text",
            "description",
            "category",
            "suitable_for"
        ]:
            text_parts.append(str(row.get(column, "")))

    return " ".join(text_parts)

In [6]:
def evaluate_recommender(
    model_name,
    recommendation_function,
    k_values=(5, 10)
):
    results = {k: [] for k in k_values}
    evaluated_users = 0

    for user_id, user_test in test_set.groupby("reviewer_name"):

        interests = build_user_interests(
            user_id,
            train_set,
            places
        )

        if not interests.strip():
            continue

        relevant_places = set(
            user_test["place_id"].astype(str)
        )

        recommendations = recommendation_function(
            user_id,
            interests,
            len(places)
        )

        if recommendations is None or recommendations.empty:
            recommended_ids = []
        else:
            seen_places = set(
                train_set.loc[
                    train_set["reviewer_name"] == user_id,
                    "place_id"
                ].astype(str)
            )

            recommendations = recommendations[
                ~recommendations["place_id"].astype(str).isin(
                    seen_places
                )
            ]

            recommended_ids = (
                recommendations["place_id"].astype(str).tolist()
            )

        evaluated_users += 1

        for k in k_values:
            top_k = recommended_ids[:k]

            hits = len(
                set(top_k) & relevant_places
            )

            results[k].append(hits / k)

    output = {
        "Model": model_name,
        "Users evaluated": evaluated_users
    }

    for k in k_values:
        output[f"Precision@{k}"] = (
            np.mean(results[k]) if results[k] else 0.0
        )

    return output

In [8]:
# Content-based recommendations
def recommend_content(user_id, interests, top_n=10):
    return content_model.recommend(
        interests=interests,
        top_n=top_n
    )


# Collaborative recommendations
def recommend_collaborative(user_id, interests, top_n=10):
    return collaborative_model.recommend_for_user(
        user_id=user_id,
        top_n=top_n
    )


# Hybrid recommendations (70% content + 30% collaborative)
def recommend_hybrid(user_id, interests, top_n=10):
    return hybrid_model.recommend(
        user_id=user_id,
        interests=interests,
        top_n=top_n,
        content_weight=0.7,
        collaborative_weight=0.3
    )

print("All three recommendation functions are defined!")

All three recommendation functions are defined!


In [9]:
content_results = evaluate_recommender(
    "Content-based",
    recommend_content
)

collaborative_results = evaluate_recommender(
    "Collaborative",
    recommend_collaborative
)

hybrid_results = evaluate_recommender(
    "Hybrid (70/30)",
    recommend_hybrid
)

evaluation_results = pd.DataFrame([
    content_results,
    collaborative_results,
    hybrid_results
])

display(evaluation_results.round(4))

,Model,Users evaluated,Precision@5,Precision@10
0,Content-based,70,0.0086,0.0086
1,Collaborative,70,0.0143,0.0100
2,Hybrid (70/30),70,0.0114,0.0100


In [10]:
evaluation_results.to_csv(
    "/content/tripsense_precision_results.csv",
    index=False
)

print("Evaluation results saved successfully!")

display(evaluation_results)

Evaluation results saved successfully!


,Model,Users evaluated,Precision@5,Precision@10
0,Content-based,70,0.008571,0.008571
1,Collaborative,70,0.014286,0.010000
2,Hybrid (70/30),70,0.011429,0.010000


In [ ]:
# ## Results and Discussion

# The recommendation models were evaluated using Precision@5
# and Precision@10.

# Collaborative filtering achieved the highest Precision@5 in
# the observed evaluation results. It also matched the hybrid
# model at Precision@10.

# The hybrid model combines destination content with patterns
# from user ratings. However, the observed results do not
# demonstrate that it outperforms collaborative filtering.

# ## Limitations

# 1. The review dataset is synthetic.
# 2. The evaluation measures relevance using ratings of 4 or 5.
# 3. Results may change with different train/test splits.
# 4. Precision@K alone does not measure recommendation diversity.
# 5. Real-world performance requires validation using genuine
#    user interactions.

# ## Conclusion

# The prototype demonstrates content-based, collaborative,
# and hybrid recommendation techniques for TripSense.
# Further evaluation is needed before deployment.